# `safim_eval_1000` — Phase A: build the fixed evaluation set (CPU)
**Kaggle notebook — thin controller only. All logic lives in `scripts/generate_safim_eval_1000.py` and `src/fim/`.**

This is the **first** of the two `safim_eval_1000` notebooks. It only *creates the evaluation dataset* — the model comparison lives in `safim_eval_1000_compare.ipynb`, which you run after this one.

`scripts/generate_safim_eval_1000.py`, driven entirely by `configs/data/safim_eval_1000.yaml`, randomly selects ~750 files from `stack_v3_python_only_data_without_fim` — a small held-out pool that **continues the 10k pilot's `stack-v3-train` stream past `sample_filtered_data_10000`'s final checkpoint** (shard 9 / row 13500), so every file in it is strictly newer than anything the pilot scanned and hash-disjoint from it. It buckets those files by AST span type and samples exactly 1000 FIM tasks per the fixed quota (line=150, statement=150, expression=150, block=120, function-body=120, method-body=100, api-call=120, class-level=90), pushing the result to `experiment/safim_eval_1000/` in `the-stack-v3-python-fim-data`.

**Idempotent** — once that folder exists on HF, re-running the build cell is a no-op (pass `--force` to deliberately regenerate, which is rare). This set must stay fixed forever once it is used for a real comparison.

CPU-only — no GPU, no accelerator needed.

> **Prerequisite:** the `stack_v3_python_only_data_without_fim` pool must already have ≥ `sampling.source_file_count` files pushed to HF — run the `stack_v3_python_only_data_without_fim` notebook first (it resumes from the 10k pilot checkpoint, target 2,000 files).

In [ ]:
# ── Cell 1: Clone repository at the requested Git state ──────────────────
import os
import shutil
import subprocess

GITHUB_REPO = "https://github.com/Rudra-G-23/qwen2.5-coder-0.5b-python-fim.git"

# Set these as needed
BRANCH = "feat/data"  # None -> main
COMMIT = None  # None -> latest commit on BRANCH

REPO_DIR = "/kaggle/working/qwen2.5-coder-0.5b-python-fim"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

print(f"Cloning branch : {BRANCH or 'main'}")
print(f"Requested commit : {COMMIT or '(latest on branch)'}")

clone_cmd = ["git", "clone"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [GITHUB_REPO, REPO_DIR]
subprocess.run(clone_cmd, check=True)

if COMMIT:
    subprocess.run(["git", "-C", REPO_DIR, "checkout", COMMIT], check=True)

current_branch = subprocess.check_output(
    ["git", "-C", REPO_DIR, "branch", "--show-current"], text=True
).strip()
current_commit = subprocess.check_output(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True
).strip()

print("\n✓ Repository ready")
print(f"  Branch : {current_branch or '(detached HEAD)'}")
print(f"  Commit : {current_commit}")


In [ ]:
# ── Cell 2: Install dependencies (CPU-only — no torch needed) ───────────
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "datasets", "pyarrow", "huggingface_hub", "pyyaml"],
    check=True,
)


In [ ]:
# ── Cell 3: Authenticate to Hugging Face ──────────────
# HF_TOKEN is stored as a Kaggle Secret — NEVER hardcode tokens.
# Add it: Kaggle account → Settings → Secrets → Add New Secret
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
print("✓ HF_TOKEN loaded from Kaggle Secrets")


## Build the fixed `safim_eval_1000` set (idempotent)

In [ ]:
# ── Cell 4: Build the fixed eval set (no-ops if it already exists on HF) ──
import subprocess
import sys

os.chdir(REPO_DIR)

subprocess.run(
    [sys.executable, "scripts/generate_safim_eval_1000.py",
     "--config", "configs/data/safim_eval_1000.yaml"],
    check=True,
)


In [ ]:
# ── Cell 5: Show the pushed eval set's metadata (bucket counts, shortfall) ──
import json
from huggingface_hub import hf_hub_download

meta_path = hf_hub_download(
    repo_id="Rudra-G-23/the-stack-v3-python-fim-data",
    filename="experiment/safim_eval_1000/metadata.json",
    repo_type="dataset",
    token=os.environ["HF_TOKEN"],
)
with open(meta_path, encoding="utf-8") as f:
    print(json.dumps(json.load(f), indent=2))

print("\n" + "=" * 70)
print("Eval set ready. Next: run safim_eval_1000_compare.ipynb on a GPU\n"
      "session to score Base / Random-LoRA / Distributed-LoRA on these\n"
      "same 1000 tasks and log the comparison graphs to W&B.")
